In [ ]:
import pandas as pd
import numpy as np
import json
import os
from scipy.stats import chi2_contingency
from IPython.display import display

# ==========================================
# 1. 데이터 로드 및 병합
# ==========================================
base_path = '../../../../data/preprocessed/'

df_graded = pd.read_csv(os.path.join(base_path, 'steam_indie_games_graded.csv'))
df_silence = pd.read_csv(os.path.join(base_path, 'steam_indie_games_silence.csv'))

df = pd.concat([df_graded, df_silence], ignore_index=True)
df.drop_duplicates(subset=['appid'], inplace=True)

df = df[df['total_reviews'] > 0].copy()
df['stage'] = df['total_reviews'].apply(lambda x: 1 if 1 <= x <= 9 else (2 if 10 <= x <= 49 else 3))

# ==========================================
# 2. 태그 리스트 추출
# ==========================================
def extract_tags(tag_str):
    if pd.isna(tag_str):
        return []
    try:
        tag_dict = json.loads(tag_str.replace('""', '"'))
        return list(tag_dict.keys())
    except:
        return []

df['tag_list'] = df['tags'].apply(extract_tags)

# ==========================================
# 3. 🚨 [핵심 수정] '게임 단위'로 교차표 및 오즈비 계산
# ==========================================
# 스테이지별 전체 게임 수
total_games_s1 = len(df[df['stage'] == 1])
total_games_s2 = len(df[df['stage'] == 2])

# 전체 고유 태그 목록 추출
all_tags_s1 = [tag for sublist in df[df['stage'] == 1]['tag_list'] for tag in sublist]
all_tags_s2 = [tag for sublist in df[df['stage'] == 2]['tag_list'] for tag in sublist]
all_unique_tags = set(all_tags_s1 + all_tags_s2)

comparison = []

for tag in all_unique_tags:
    # 1단계에서 이 태그를 보유한 게임 수 / 미보유 게임 수
    games_s1_with = df[df['stage'] == 1]['tag_list'].apply(lambda tags: tag in tags).sum()
    games_s1_without = total_games_s1 - games_s1_with
    
    # 2단계에서 이 태그를 보유한 게임 수 / 미보유 게임 수
    games_s2_with = df[df['stage'] == 2]['tag_list'].apply(lambda tags: tag in tags).sum()
    games_s2_without = total_games_s2 - games_s2_with
    
    # 게임 보유율(%) 계산
    rate_s1 = (games_s1_with / total_games_s1) * 100 if total_games_s1 > 0 else 0
    rate_s2 = (games_s2_with / total_games_s2) * 100 if total_games_s2 > 0 else 0
    
    # 2x2 교차표 (게임 수 기준)
    table = [[games_s1_with, games_s1_without],
             [games_s2_with, games_s2_without]]
    
    # 카이제곱 검정
    try:
        chi2, p, _, _ = chi2_contingency(table)
    except ValueError:
        p = np.nan
        
    # 오즈비 계산
    try:
        if (games_s1_with * games_s2_without) > 0:
            odds_ratio = (games_s2_with * games_s1_without) / (games_s1_with * games_s2_without) 
        else:
            odds_ratio = np.nan
    except ZeroDivisionError:
        odds_ratio = np.nan

    p_str = "< 0.001" if pd.notna(p) and p < 0.001 else f"{p:.4f}"

    comparison.append({
        'Tag': tag,
        'Stage1_보유율(%)': rate_s1,
        'Stage2_보유율(%)': rate_s2,
        'Growth(%p)': rate_s2 - rate_s1,
        'Odds_Ratio(배)': odds_ratio,
        'P-value': p_str,
        'Significant': '✅' if pd.notna(p) and p < 0.05 else '❌'
    })

comp_df = pd.DataFrame(comparison)

# ==========================================
# 4. 결과 출력 (이미지 피드백 전면 반영)
# ==========================================
# 노이즈 제거: 이미지 조언대로 'Stage 2 게임 보유율이 0.5% 이상'인 경우로 필터링
comp_df = comp_df[comp_df['Stage2_보유율(%)'] >= 0.5]

# 오즈비(Odds Ratio) 기준으로 정렬
rising_top10 = comp_df.sort_values('Odds_Ratio(배)', ascending=False).head(10)
trap_top10 = comp_df.sort_values('Odds_Ratio(배)', ascending=True).head(10)

print("💡 [성장 동력] '게임 수' 기준 오즈비가 가장 높은 태그 (1단계 -> 2단계 진입 확률 UP)")
display(rising_top10[['Tag', 'Stage1_보유율(%)', 'Stage2_보유율(%)', 'Odds_Ratio(배)', 'P-value', 'Significant']].round(3))

print("\n💀 [함정 태그] '게임 수' 기준 오즈비가 가장 낮은 태그 (1단계 정체 확률 UP)")
display(trap_top10[['Tag', 'Stage1_보유율(%)', 'Stage2_보유율(%)', 'Odds_Ratio(배)', 'P-value', 'Significant']].round(3))

💡 [성장 동력] '게임 수' 기준 오즈비가 가장 높은 태그 (1단계 -> 2단계 진입 확률 UP)


,Tag,Stage1_보유율(%),Stage2_보유율(%),Odds_Ratio(배),P-value,Significant
97,Blood,0.306,0.558,1.828,0.0544,❌
31,Competitive,0.413,0.702,1.706,0.0499,✅
379,Roguelike Deckbuilder,0.474,0.723,1.529,0.1087,❌
201,Fast-Paced,1.025,1.488,1.458,0.0327,✅
293,Metroidvania,1.514,2.066,1.372,0.0317,✅
41,Steampunk,0.444,0.599,1.353,0.3081,❌
194,Cozy,0.918,1.219,1.332,0.1419,❌
282,Replay Value,1.162,1.488,1.284,0.1528,❌
202,Soundtrack,0.642,0.806,1.256,0.3617,❌
241,LGBTQ+,1.239,1.550,1.255,0.1844,❌



💀 [함정 태그] '게임 수' 기준 오즈비가 가장 낮은 태그 (1단계 정체 확률 UP)


,Tag,Stage1_보유율(%),Stage2_보유율(%),Odds_Ratio(배),P-value,Significant
157,Singleplayer,79.367,47.521,0.235,< 0.001,✅
96,Indie,68.691,38.409,0.284,< 0.001,✅
55,3D Vision,1.591,0.517,0.321,< 0.001,✅
105,Casual,52.279,27.624,0.348,< 0.001,✅
339,Match 3,1.606,0.579,0.356,< 0.001,✅
348,Education,4.084,1.570,0.375,< 0.001,✅
222,3D Platformer,9.712,4.153,0.403,< 0.001,✅
409,PvE,12.481,5.744,0.427,< 0.001,✅
57,3D,36.280,20.289,0.447,< 0.001,✅
304,2D,46.421,28.058,0.450,< 0.001,✅


p벨류 유의미한에들만

In [2]:
import pandas as pd
import numpy as np
import json
import os
from scipy.stats import chi2_contingency
from IPython.display import display

# ==========================================
# 1. 데이터 로드 및 병합
# ==========================================
base_path = '../../../../data/preprocessed/'

# 파일 불러오기
df_graded = pd.read_csv(os.path.join(base_path, 'steam_indie_games_graded.csv'))
df_silence = pd.read_csv(os.path.join(base_path, 'steam_indie_games_silence.csv'))

# 데이터 합치기 및 중복 제거
df = pd.concat([df_graded, df_silence], ignore_index=True)
df.drop_duplicates(subset=['appid'], inplace=True)

# 리뷰가 1개 이상인 게임(1, 2단계)만 필터링
df = df[df['total_reviews'] > 0].copy()
df['stage'] = df['total_reviews'].apply(lambda x: 1 if 1 <= x <= 9 else (2 if 10 <= x <= 49 else 3))

# ==========================================
# 2. 태그 리스트 추출
# ==========================================
def extract_tags(tag_str):
    if pd.isna(tag_str):
        return []
    try:
        tag_dict = json.loads(tag_str.replace('""', '"'))
        return list(tag_dict.keys())
    except:
        return []

df['tag_list'] = df['tags'].apply(extract_tags)

# ==========================================
# 3. '게임 단위'로 교차표 및 오즈비 계산
# ==========================================
# 스테이지별 전체 게임 수
total_games_s1 = len(df[df['stage'] == 1])
total_games_s2 = len(df[df['stage'] == 2])

# 전체 고유 태그 목록 추출
all_tags_s1 = [tag for sublist in df[df['stage'] == 1]['tag_list'] for tag in sublist]
all_tags_s2 = [tag for sublist in df[df['stage'] == 2]['tag_list'] for tag in sublist]
all_unique_tags = set(all_tags_s1 + all_tags_s2)

comparison = []

for tag in all_unique_tags:
    # 1단계에서 이 태그를 보유한 게임 수 / 미보유 게임 수
    games_s1_with = df[df['stage'] == 1]['tag_list'].apply(lambda tags: tag in tags).sum()
    games_s1_without = total_games_s1 - games_s1_with
    
    # 2단계에서 이 태그를 보유한 게임 수 / 미보유 게임 수
    games_s2_with = df[df['stage'] == 2]['tag_list'].apply(lambda tags: tag in tags).sum()
    games_s2_without = total_games_s2 - games_s2_with
    
    # 게임 보유율(%) 계산
    rate_s1 = (games_s1_with / total_games_s1) * 100 if total_games_s1 > 0 else 0
    rate_s2 = (games_s2_with / total_games_s2) * 100 if total_games_s2 > 0 else 0
    
    # 2x2 교차표 (게임 수 기준)
    table = [[games_s1_with, games_s1_without],
             [games_s2_with, games_s2_without]]
    
    # 카이제곱 검정
    try:
        chi2, p, _, _ = chi2_contingency(table)
    except ValueError:
        p = np.nan
        
    # 오즈비 계산
    try:
        if (games_s1_with * games_s2_without) > 0:
            odds_ratio = (games_s2_with * games_s1_without) / (games_s1_with * games_s2_without) 
        else:
            odds_ratio = np.nan
    except ZeroDivisionError:
        odds_ratio = np.nan

    p_str = "< 0.001" if pd.notna(p) and p < 0.001 else f"{p:.4f}"

    comparison.append({
        'Tag': tag,
        'Stage1_보유율(%)': rate_s1,
        'Stage2_보유율(%)': rate_s2,
        'Growth(%p)': rate_s2 - rate_s1,
        'Odds_Ratio(배)': odds_ratio,
        'P-value': p_str,
        'Significant': '✅' if pd.notna(p) and p < 0.05 else '❌'
    })

comp_df = pd.DataFrame(comparison)

# ==========================================
# 4. 🚨 [최종 필터링 및 Top 10 추출]
# ==========================================
# 1차 필터링: 노이즈 제거 (Stage 2 게임 보유율 0.5% 이상)
comp_df = comp_df[comp_df['Stage2_보유율(%)'] >= 0.5]

# 2차 필터링: 깐깐한 통계 검정을 통과한 유의미한 태그만 추출 (✅ 인 경우만)
valid_tags = comp_df[comp_df['Significant'] == '✅']

# 💡 오즈비(Odds Ratio) 기준으로 정렬하여 찐 Top 10 추출
rising_top10 = valid_tags.sort_values('Odds_Ratio(배)', ascending=False).head(10)
trap_top10 = valid_tags.sort_values('Odds_Ratio(배)', ascending=True).head(10)

print("💡 [성장 동력] '게임 수' 기준 오즈비가 가장 높은 태그 (통계적 유의성 100% 검증 완료)")
display(rising_top10[['Tag', 'Stage1_보유율(%)', 'Stage2_보유율(%)', 'Odds_Ratio(배)', 'P-value', 'Significant']].round(3))

print("\n💀 [함정 태그] '게임 수' 기준 오즈비가 가장 낮은 태그 (통계적 유의성 100% 검증 완료)")
display(trap_top10[['Tag', 'Stage1_보유율(%)', 'Stage2_보유율(%)', 'Odds_Ratio(배)', 'P-value', 'Significant']].round(3))

💡 [성장 동력] '게임 수' 기준 오즈비가 가장 높은 태그 (통계적 유의성 100% 검증 완료)


,Tag,Stage1_보유율(%),Stage2_보유율(%),Odds_Ratio(배),P-value,Significant
31,Competitive,0.413,0.702,1.706,0.0499,✅
201,Fast-Paced,1.025,1.488,1.458,0.0327,✅
293,Metroidvania,1.514,2.066,1.372,0.0317,✅
292,Story Rich,14.775,12.893,0.854,0.0046,✅
294,Procedural Generation,5.124,4.277,0.827,0.0401,✅
426,Walking Simulator,4.925,4.008,0.806,0.0227,✅
355,Multiple Endings,8.168,6.632,0.799,0.0024,✅
392,Management,4.420,3.512,0.787,0.0171,✅
64,Interactive Fiction,5.231,4.153,0.785,0.0087,✅
207,Female Protagonist,8.152,6.446,0.776,< 0.001,✅



💀 [함정 태그] '게임 수' 기준 오즈비가 가장 낮은 태그 (통계적 유의성 100% 검증 완료)


,Tag,Stage1_보유율(%),Stage2_보유율(%),Odds_Ratio(배),P-value,Significant
157,Singleplayer,79.367,47.521,0.235,< 0.001,✅
96,Indie,68.691,38.409,0.284,< 0.001,✅
55,3D Vision,1.591,0.517,0.321,< 0.001,✅
105,Casual,52.279,27.624,0.348,< 0.001,✅
339,Match 3,1.606,0.579,0.356,< 0.001,✅
348,Education,4.084,1.570,0.375,< 0.001,✅
222,3D Platformer,9.712,4.153,0.403,< 0.001,✅
409,PvE,12.481,5.744,0.427,< 0.001,✅
57,3D,36.280,20.289,0.447,< 0.001,✅
304,2D,46.421,28.058,0.450,< 0.001,✅
